In [0]:
df_spark = spark.read.csv(
    '/Volumes/workspace/default/trafficdata/traffic.csv',
    header=True,
    inferSchema=True
)
display(df_spark)

In [0]:
df_spark.head(5)

In [0]:
print(df_spark.columns)
print(df_spark.dtypes)

In [0]:
print(df_spark.count())
df_spark.printSchema()
df_spark.describe().show()

In [0]:
print(df_spark.isEmpty())

In [0]:
from pyspark.sql.functions import hour, dayofweek, col, when

# Sort the DataFrame by 'Junction' and 'DateTime'
df_sorted = df_spark.orderBy(['Junction', 'DateTime'])

# Extract core time features
df_features = (
    df_sorted
    .withColumn('hour', hour(col('DateTime')))
    .withColumn('day_of_week', dayofweek(col('DateTime')) - 2)  # Spark: Monday=2, Sunday=1
    .withColumn('is_weekend', when((col('day_of_week') > 4), 1).otherwise(0))
)

display(df_features.show(5))

In [0]:
print("Isolation Forest Anomaly Detection (Unsupervised)")

In [0]:
# If needed, convert Spark DataFrame to pandas (takes a sample for speed)
pandas_sample = df_features.sample(fraction=0.1, seed=42).toPandas()  # Adjust fraction as needed

# Select features for anomaly detection
feature_cols = ['Vehicles', 'hour', 'day_of_week', 'is_weekend']
X = pandas_sample[feature_cols]

In [0]:
from sklearn.ensemble import IsolationForest

# Initialize Isolation Forest
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
iso_forest.fit(X)

# Predict anomalies
pandas_sample['anomaly'] = iso_forest.predict(X)   # -1 = anomaly, 1 = normal
pandas_sample['anomaly_score'] = iso_forest.decision_function(X)

# Show a few rows, highlight anomalies
print(pandas_sample[['DateTime', 'Junction', 'Vehicles', 'hour', 'day_of_week', 'is_weekend', 'anomaly', 'anomaly_score']].head(10))

# How many anomalies detected?
print("Anomalies detected:", (pandas_sample['anomaly'] == -1).sum())


In [0]:
# How many anomalies?
n_anomalies = (pandas_sample['anomaly'] == -1).sum()
n_total = len(pandas_sample)
print(f"Anomalies detected: {n_anomalies} out of {n_total} ({100*n_anomalies/n_total:.2f}%)")

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.hist(pandas_sample[pandas_sample['anomaly'] == 1]['Vehicles'], bins=30, alpha=0.5, label='Normal')
plt.hist(pandas_sample[pandas_sample['anomaly'] == -1]['Vehicles'], bins=30, alpha=0.7, label='Anomaly', color='red')
plt.legend()
plt.xlabel('Vehicles Count')
plt.ylabel('Frequency')
plt.title('Vehicle Count Distribution: Normal vs Anomaly')
plt.show()


In [0]:
import matplotlib.pyplot as plt

# Example for a single junction (e.g., Junction 1)
dfj = pandas_sample[pandas_sample['Junction'] == 1].sort_values('DateTime')

plt.figure(figsize=(14,5))
plt.plot(dfj['DateTime'], dfj['Vehicles'], label='Vehicle Count', color='blue', linewidth=1)
plt.scatter(dfj[dfj['anomaly']==-1]['DateTime'],
            dfj[dfj['anomaly']==-1]['Vehicles'],
            color='red', s=30, label='Anomaly')
plt.xlabel('DateTime')
plt.ylabel('Vehicles')
plt.title('Traffic Count With Anomalies (Junction 1)')
plt.legend()
plt.show()


In [0]:
print("Time-Series Forecasting for Anomaly Detection")

In [0]:
# Pick one junction for time-series modeling, e.g., Junction 1
df_single = pandas_sample[pandas_sample['Junction'] == 1].copy()
df_single = df_single.sort_values('DateTime')

In [0]:
# Prophet requires 'ds' (date) and 'y' (value)
df_prophet = df_single[['DateTime', 'Vehicles']].rename(columns={'DateTime': 'ds', 'Vehicles': 'y'})

In [0]:
from prophet import Prophet

model = Prophet()
model.fit(df_prophet)

# Forecast/simulate the same periods
future = df_prophet[['ds']]
forecast = model.predict(future)
df_prophet['yhat'] = forecast['yhat']
df_prophet['residual'] = df_prophet['y'] - df_prophet['yhat']

In [0]:
import numpy as np

# Use 3 standard deviations as anomaly threshold (adjust for your use case)
std_resid = df_prophet['residual'].std()
df_prophet['ts_anomaly'] = np.where(np.abs(df_prophet['residual']) > 3*std_resid, -1, 1)

# Show results
print(df_prophet[['ds', 'y', 'yhat', 'residual', 'ts_anomaly']].head(10))
print("Time-series anomalies detected:", (df_prophet['ts_anomaly'] == -1).sum())


In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(df_prophet['ds'], df_prophet['y'], label='Actual', alpha=0.6)
plt.plot(df_prophet['ds'], df_prophet['yhat'], label='Forecast', alpha=0.7)
plt.scatter(df_prophet.loc[df_prophet['ts_anomaly']==-1, 'ds'],
            df_prophet.loc[df_prophet['ts_anomaly']==-1, 'y'],
            color='red', label='Anomaly')
plt.legend()
plt.xlabel('DateTime')
plt.ylabel('Vehicles')
plt.title('Prophet Forecast vs Actual with Anomalies')
plt.show()


In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

features = ['Vehicles', 'hour', 'day_of_week', 'Junction']
X = pandas_sample[features].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Run DBSCAN
dbscan = DBSCAN(eps=0.75, min_samples=8)
dbscan.fit(X_scaled)

# Assign cluster labels
pandas_sample['dbscan_cluster'] = dbscan.labels_

# Cluster summary
print(pandas_sample['dbscan_cluster'].value_counts())
print(pandas_sample.groupby(['dbscan_cluster', 'anomaly']).size())


In [0]:
import matplotlib.pyplot as plt

# Sort by DateTime for proper line plot
df_plot = pandas_sample.sort_values('DateTime')

plt.figure(figsize=(15,6))

# For each cluster, plot vehicles as a separate line (clusters may be int)
clusters = df_plot['dbscan_cluster'].unique()
for cluster_id in clusters:
    cluster_data = df_plot[df_plot['dbscan_cluster'] == cluster_id]
    plt.plot(cluster_data['DateTime'], cluster_data['Vehicles'], label=f'Cluster {cluster_id}')

plt.xlabel('DateTime')
plt.ylabel('Vehicles')
plt.title('Vehicle Flow Over Time by DBSCAN Cluster')
plt.legend()
plt.tight_layout()
plt.show()


In [0]:
import mlflow
import matplotlib.pyplot as plt

with mlflow.start_run(run_name="IsolationForest"):
    mlflow.log_param("contamination", 0.05)
    mlflow.log_metric("anomalies_detected", (pandas_sample['anomaly'] == -1).sum())
    
    # Plot: Vehicles over time, anomalies in red (for Junction 1)
    junction_id = 1
    df_junc = pandas_sample[pandas_sample['Junction'] == junction_id].sort_values('DateTime')
    
    plt.figure(figsize=(12,5))
    plt.plot(df_junc['DateTime'], df_junc['Vehicles'], label='Vehicles', color='blue')
    plt.scatter(df_junc[df_junc['anomaly'] == -1]['DateTime'],
                df_junc[df_junc['anomaly'] == -1]['Vehicles'],
                color='red', label='Anomaly', s=40)
    plt.xlabel('DateTime')
    plt.ylabel('Vehicles')
    plt.title(f'Isolation Forest Anomalies (Junction {junction_id})')
    plt.legend()
    plt.tight_layout()
    plt.savefig("isoforest_line.png")
    mlflow.log_artifact("isoforest_line.png")
    plt.close()


In [0]:
import mlflow
import matplotlib.pyplot as plt

with mlflow.start_run(run_name="ProphetForecast"):
    mlflow.log_param("model", "Prophet")
    mlflow.log_metric("n_anomalies_prophet", (df_prophet['ts_anomaly'] == -1).sum())
    mlflow.log_metric("mean_abs_error", (df_prophet['y'] - df_prophet['yhat']).abs().mean())
    
    plt.figure(figsize=(12,5))
    plt.plot(df_prophet['ds'], df_prophet['y'], label='Actual', color='blue')
    plt.plot(df_prophet['ds'], df_prophet['yhat'], label='Forecast', color='green')
    plt.scatter(df_prophet[df_prophet['ts_anomaly'] == -1]['ds'],
                df_prophet[df_prophet['ts_anomaly'] == -1]['y'],
                color='red', label='Anomaly', s=40)
    plt.xlabel('DateTime')
    plt.ylabel('Vehicles')
    plt.title('Prophet: Forecast vs Actual with Anomalies')
    plt.legend()
    plt.tight_layout()
    plt.savefig("prophet_line.png")
    mlflow.log_artifact("prophet_line.png")
    plt.close()


In [0]:
import mlflow
import matplotlib.pyplot as plt

with mlflow.start_run(run_name="DBSCANClustering"):
    mlflow.log_param("model", "DBSCAN")
    mlflow.log_param("eps", 0.75)
    mlflow.log_param("min_samples", 8)
    mlflow.log_metric("n_clusters", len(set(pandas_sample['dbscan_cluster'])) - (1 if -1 in pandas_sample['dbscan_cluster'].values else 0))
    mlflow.log_metric("n_noise", (pandas_sample['dbscan_cluster'] == -1).sum())
    
    plt.figure(figsize=(15,6))
    for cluster_id in pandas_sample['dbscan_cluster'].unique():
        cluster_data = pandas_sample[pandas_sample['dbscan_cluster'] == cluster_id].sort_values('DateTime')
        label = f'Cluster {cluster_id}'
        plt.plot(cluster_data['DateTime'], cluster_data['Vehicles'], label=label)
    plt.xlabel('DateTime')
    plt.ylabel('Vehicles')
    plt.title('DBSCAN: Vehicle Flow Over Time by Cluster')
    plt.legend()
    plt.tight_layout()
    plt.savefig("dbscan_line.png")
    mlflow.log_artifact("dbscan_line.png")
    plt.close()


In [0]:
# For each junction, calculate rolling mean and change in vehicles
pandas_sample = pandas_sample.sort_values(['Junction', 'DateTime'])

# Rolling window and delta features (using groupby for each junction)
pandas_sample['rolling_mean_3h'] = pandas_sample.groupby('Junction')['Vehicles'].rolling(window=3, min_periods=1).mean().reset_index(0, drop=True)
pandas_sample['vehicle_delta'] = pandas_sample.groupby('Junction')['Vehicles'].diff().fillna(0)


In [0]:
# Dynamically build the list of available anomaly flag columns
flags = [col for col in ['anomaly', 'ts_anomaly', 'dbscan_anomaly'] if col in pandas_sample.columns]

# If using DBSCAN: treat cluster -1 as anomaly
if 'dbscan_cluster' in pandas_sample:
    pandas_sample['dbscan_anomaly'] = pandas_sample['dbscan_cluster'].apply(
        lambda x: -1 if x == -1 else 1
    )

# Example simple "ensemble anomaly" flag by majority vote
pandas_sample['ensemble_anomaly'] = pandas_sample[flags].sum(axis=1)
# Mark as anomaly if at least half or more of the models flag as anomaly
pandas_sample['ensemble_anomaly'] = pandas_sample['ensemble_anomaly'].apply(
    lambda x: -1 if x <= -1 * (len(flags) // 2) else 1
)

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

# Heatmap: count of anomalies by day_of_week and hour
heatmap_data = pandas_sample[pandas_sample['ensemble_anomaly'] == -1].pivot_table(
    index='day_of_week', columns='hour', values='Vehicles', aggfunc='count', fill_value=0)

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='Reds')
plt.title("Anomaly Count Heatmap (Day of Week vs. Hour)")
plt.xlabel('Hour')
plt.ylabel('Day of Week (0=Mon)')
plt.show()


In [0]:
import pandas as pd
import datetime

last_time = pandas_sample['DateTime'].max()
recent = pandas_sample[
    (pandas_sample['DateTime'] >= last_time - pd.Timedelta(hours=1)) &
    (pandas_sample['Junction'] == 1)
]
anomaly_count = (recent['ensemble_anomaly'] == -1).sum()

threshold = 2
if anomaly_count >= threshold:
    print(
        f"ALERT: {anomaly_count} anomalies detected at Junction 1 in the last hour!"
    )
else:
    print("No alert – anomaly count within normal range.")

In [0]:
# Aggregate per junction, sort by anomaly count
anomaly_summary = pandas_sample.groupby('Junction')['ensemble_anomaly'].apply(lambda x: (x == -1).sum()).sort_values(ascending=False)
print("Anomaly count per junction:\n", anomaly_summary)

# When do most anomalies occur?
anomalies_by_time = pandas_sample[pandas_sample['ensemble_anomaly'] == -1].groupby('hour').size()
print("Anomalies by hour of day:\n", anomalies_by_time)

# Todo: Add markdown cells summarizing how these could assist city planners (example below)
